# QCC v27 | 4,030-Toffoli coded-word chemistry

**Local geometric mean 27.31511** at the default salt (all three tasks error-free). This version changes the chemistry:
our coded-word table with 4,030-Toffoli loaders, **cost 294,888,332.10**, below every public chemistry. invsqrt and DQI
are the files of the current public frontier, pinned by SHA-256.

| Task | File | Q | Cost | Reference / cost |
|---|---|---:|---:|---:|
| invsqrt | `1ba197b0` (46-qubit line with the dead-write fix, public) | 46 | 78,673.57 | 26.3972 |
| DQI | `821bf8d1` (Syed Asad Ali's v12 + extractor cutoff 18 + retained products, public) | 889 | 360,209,905,486 | 67.8478 |
| **Chemistry** | **table `4df35abd`, coded words, 4,030-Toffoli loaders (ours)** | 427 | **294,888,332.10** | **11.3792** |

## What is ours: the chemistry
Published in our dataset *qcc-chemistry-4030-toffoli-loaders* (CC0); the earlier 4,033, 4,038, 4,044 and 4,057 sets
stay in their own datasets.
- **Coded words.** Strong Reed–Muller codes on every batch from 3 to 14 (bit 0 of each word affine in the four low address
  bits; the XOR of the five words of a batch affine at bit 1 and of degree at most 2 at bit 2), with a full 75-column
  top-bit gauge.
- **The table** was refit around those codes: λ_eff 46.9097265, error_1 19.99992 mHa, error_2 19.99999 mHa, E_gap at its
  floor.
- **The loaders:** k0 291, k1–k3 286, five loaders at 263 and six at 261 Toffolis, 4,030 per pass, so T = 14,722. Every
  loader passes the forward and reverse reachable-state checks on four seeds, leaves no scratch dirty, and applies R only
  to provably-|0> qubits.

## Checks (official verifiers)
- Chemistry: PASS, cost 294,888,332.10 (λ_eff 46.9097265, T 14,722, Q 427).
- invsqrt `1ba197b0`: PASS at the default salt and `s1` (78,673.57).
- DQI `821bf8d1`: 0 failures at the default salt, `s1` and `s2` (360,209,905,486 at the default salt).

## Credits
- **invsqrt:** Syed Asad Ali's generator (Apache-2.0); 64-row table build by Ali Haydar Özdağ (ahoirg); angle refit by
  Ertuğrul Özer (ozertuu); measured-CZ / dead-write fixes by Asterios Panoras and James Barlow (manwithacat); taken from
  manwithacat's notebook output.
- **DQI:** Syed Asad Ali's v12 decoder (Apache-2.0); extractor cutoff by Ertuğrul Özer; cutoff 18 and retained products
  by Areeb Ali (areebbb); taken from ahoirg's dataset.
- **Chemistry lineage:** ozertuu's *quantum-sim-linalg-cache* (CC0) table line and prvsiyan's refit; loader design after
  Syed Asad Ali, Lera/marculera's DF-THC pipeline and word codes, Ahmed Berat Özer's progressive rounding and Atakan
  Aldemir's π-gauge; James Barlow / manwithacat's dfthc_refit (Apache-2.0); busyaprime and Heliosli.
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran.

The code and our chemistry here are Apache-2.0 / CC0. Each other circuit keeps its authors' terms.

In [1]:
import base64, hashlib, json, os, shutil, subprocess, sys, zipfile, zlib
from pathlib import Path
import numpy as np

INPUT = Path(os.environ.get('QCC_INPUT', '/kaggle/input'))
WORK = Path(os.environ.get('QCC_WORKING', '/kaggle/working'))
TMP = Path(os.environ.get('QCC_TMP', '/tmp/qcc_chem'))

def walk(name):
    return [Path(d) / name for d, _, files in os.walk(INPUT, followlinks=True) if name in files]

COMP = next(p for p in walk('verify_submission_invsqrt.py') if 'verifiers' in str(p)).parents[2]
print('competition data:', COMP)

def sha_stream(f):
    h = hashlib.sha256()
    for block in iter(lambda: f.read(1 << 22), b''):
        h.update(block)
    return h.hexdigest()

competition data: /kaggle/input/competitions/google-verifiable-quantum-circuit-compilation


## Source files

In [2]:
# Exact public files this release uses: (SHA-256, size in bytes).
PINNED = {
    'invsqrt.kmb': ('1ba197b027713aac164056d12505c87f286c3fcd94f403c522073325763bf2f9', 198631),
    'dqi.kmb': ('821bf8d157c03c3e868a0099b0ab1438cfba664ab1bc6fec3210654099b672a3', 455584321),
    'dfthcsos/submission.npz': ('4df35abdeec320a3f1c10f6f533e73550e50bb45d95e666deaa01062810a3559', 1847692),
    'dfthcsos/angle_k_0.kmb': ('71735b23037e8a64d16e39b3347d440b36bdf8813613ec70c36a85bfd362d7c2', 410442),
    'dfthcsos/angle_k_1.kmb': ('1a43bbc368785aa5fc5210f6e4d5d2f9ddf114f7f3f6d3e3964f9478056bb4be', 420614),
    'dfthcsos/angle_k_2.kmb': ('0970b392fb4bed09d6026b77b4b31510573123cb76955030115507bffde4a6bb', 418801),
    'dfthcsos/angle_k_3.kmb': ('f9d313326dd07028bf6ef672bb47013f01e9311695d66b8d1ee2546d7dbacd7f', 421008),
    'dfthcsos/angle_k_4.kmb': ('0ab29f06e44ef247035c971dcca30e27616bdfdc2a102fbb58c3882bbfbfd1e0', 347635),
    'dfthcsos/angle_k_5.kmb': ('b346df2edafbaf4d95192a974db27abf71b770f51ee61dae2eb352f422cefdf4', 351040),
    'dfthcsos/angle_k_6.kmb': ('749d9edb817c59f8a3507400002e75c2d8db55932a6933db5c5f6f59a2679f7b', 347703),
    'dfthcsos/angle_k_7.kmb': ('0ad7d3775a36d7c14e3d4dfa4a144267e4e9759521fae4022715be25c309638e', 350049),
    'dfthcsos/angle_k_8.kmb': ('46e0f8d69e532438d476aa8244f257a649fdc4cffe9d41f9921758f4dc18fa83', 346523),
    'dfthcsos/angle_k_9.kmb': ('0b995537e919351a705babb6294dfe3525ef52a3c40b08122d842f45df8182da', 354023),
    'dfthcsos/angle_k_10.kmb': ('5fea9f297afea7ffe74054ed1b44f92031a5f6e319a7f1c8741f86c05b234834', 347748),
    'dfthcsos/angle_k_11.kmb': ('dc425931348cc4950931c400ab19322e01495b7cf3e2d02bafabcaaf9559d28a', 350356),
    'dfthcsos/angle_k_12.kmb': ('64bbada35a382068b8dfbca0df0d781df8704aa367882cc426b76e2767f3d13f', 348617),
    'dfthcsos/angle_k_13.kmb': ('958789349187a70ab3baf426655aacd2360113f71ab9a7b49d6624bd0e20bcd6', 344270),
    'dfthcsos/angle_k_14.kmb': ('f35c803dece96ceb0ba24040e7c232a73170a3692c4f5212bdeadcbfcd154212', 352571),
}

def candidates():
    """(label, opener, basename, size) for every .kmb / submission.npz under the inputs, inside zips too."""
    for d, _, files in os.walk(INPUT, followlinks=True):
        for f in files:
            p = Path(d) / f
            base = f.replace('dfthcsos__', '')
            if base.endswith('.kmb') or base == 'submission.npz':
                yield str(p), (lambda p=p: open(p, 'rb')), base, p.stat().st_size
            elif f.endswith('.zip'):
                try:
                    z = zipfile.ZipFile(p)
                except zipfile.BadZipFile:
                    continue
                for info in z.infolist():
                    base = info.filename.rsplit('/', 1)[-1]
                    if base.endswith('.kmb') or base == 'submission.npz':
                        yield f'{p}:{info.filename}', (lambda z=z, n=info.filename: z.open(n)), base, info.file_size

if TMP.exists():
    shutil.rmtree(TMP)
SUB = TMP / 'submission'
(SUB / 'dfthcsos').mkdir(parents=True)
wanted = {(name.rsplit('/', 1)[-1], size, digest): name for name, (digest, size) in PINNED.items()}
found = {}
for label, opener, base, size in candidates():
    keys = [k for k in wanted if (k[0] == base or (k[0] == 'dqi.kmb' and base.startswith('dqi'))) and k[1] == size and wanted[k] not in found]
    if keys:
        with opener() as f:
            digest = sha_stream(f)
        for k in keys:
            if k[2] == digest:
                with opener() as f, open(SUB / wanted[k], 'wb') as g:
                    shutil.copyfileobj(f, g, 1 << 22)
                found[wanted[k]] = label
import lzma
if 'dqi.kmb' not in found:
    for xz in walk('dqi.kmb.xz'):
        data = lzma.decompress(xz.read_bytes())
        if hashlib.sha256(data).hexdigest() == PINNED['dqi.kmb'][0]:
            (SUB / 'dqi.kmb').write_bytes(data)
            found['dqi.kmb'] = str(xz)
            break
missing = sorted(set(PINNED) - set(found))
assert not missing, f'not found in the attached inputs: {missing}'
for name in PINNED:
    print(f'{name:24s} <- {found[name]}')

invsqrt.kmb              <- /kaggle/input/vqcc-hedge-wizardry-notebook/submission.zip:invsqrt.kmb
dqi.kmb                  <- /kaggle/input/datasets/ahoirg/qcc-portfolio-gm-27-28999/dqi.kmb
dfthcsos/submission.npz  <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/submission.npz
dfthcsos/angle_k_0.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angle_k_0.kmb
dfthcsos/angle_k_1.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angle_k_1.kmb
dfthcsos/angle_k_2.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angle_k_2.kmb
dfthcsos/angle_k_3.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angle_k_3.kmb
dfthcsos/angle_k_4.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angle_k_4.kmb
dfthcsos/angle_k_5.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angle_k_5.kmb
dfthcsos/angle_k_6.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angle_k_6.kmb
dfthcsos/angle_k_7.kmb   <- /kaggle/input/qcc-chemistry-4030-toffoli-loaders/angl

## Official verification

In [3]:
# Official verifiers from the competition data, unmodified, default salt and shot counts.
env = dict(os.environ, PYTHONPATH=f'{COMP / "kickmix"}:{COMP / "verifiers"}')

def run(args):
    out = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env)
    text = out.stdout
    return json.loads(text[text.index('{'):])

reports = {}
reports['dfthcsos'] = run([COMP / 'verifiers/dfthcsos/verify.py', SUB / 'dfthcsos'])
m = reports['dfthcsos']['metrics']
print('chemistry', reports['dfthcsos']['status'], reports['dfthcsos']['score'], 'lambda_eff', reports['dfthcsos']['lambda_eff'],
      'T', reports['dfthcsos']['toffoli_count'], 'error_1', m['error_1'], 'error_2', m['error_2'], 'E_gap', m['E_gap'])
reports['dqi'] = run([COMP / 'verifiers/dqi/verify_submission_dqi.py', SUB / 'dqi.kmb'])
print('DQI', reports['dqi']['status'], reports['dqi']['score'], 'errors',
      reports['dqi']['value_errors'], reports['dqi']['phase_errors'], 'Q', reports['dqi']['num_qubits'])
reports['invsqrt'] = run([COMP / 'verifiers/invsqrt/verify_submission_invsqrt.py', SUB / 'invsqrt.kmb'])
print('invsqrt', reports['invsqrt']['status'], reports['invsqrt']['score'], 'Q', reports['invsqrt']['num_qubits'],
      'worst error/tolerance', reports['invsqrt']['worst_error_over_tolerance'])

chemistry PASS 294888332.1042627 lambda_eff 46.909726478631555 T 14722.0 error_1 0.0199999200057113 error_2 0.01999998805240959 E_gap 3.914898001243273
DQI PASS 360209905486.0156 errors 0 0 Q 889
invsqrt PASS 78673.56921101942 Q 46 worst error/tolerance 0.9986660009092673


## Score and submission

In [4]:
REFERENCE = {'invsqrt': 2076774.233715, 'dqi': 24439463583672.93, 'dfthcsos': 3355599588.3155375}
assert all(r['status'] == 'PASS' for r in reports.values())
ratios = {t: REFERENCE[t] / reports[t]['score'] for t in REFERENCE}
for t, r in ratios.items():
    print(f'{t:9s} cost {reports[t]["score"]:.6f}  ratio {r:.5f}')
print('geometric mean, this run', round(float(np.prod(list(ratios.values())) ** (1 / 3)), 5))

WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(WORK / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(SUB.rglob('*')):
        if path.is_file():
            z.write(path, path.relative_to(SUB).as_posix())
print('wrote', WORK / 'submission.zip', (WORK / 'submission.zip').stat().st_size, 'bytes')
shutil.rmtree(TMP)

invsqrt   cost 78673.569211  ratio 26.39736
dqi       cost 360209905486.015625  ratio 67.84784
dfthcsos  cost 294888332.104263  ratio 11.37922
geometric mean, this run 27.31511
wrote /kaggle/working/submission.zip 49283081 bytes
